# Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

# Important libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from os.path import join
import joblib
import sys
import torch
sys.path.append("../../")

from src.configs.default_configs import fn_model, fn_pred, fn_pred_perf, device
from src.configs.lung_config import data_name
from src.file_manager.filepath import FilePath
from src.training.misc import get_pos_weight
from src.models.rue.model import AE_Predictor
from src.models.rue.training import train_classifier, train_decoder
from src.training.tuning import tune_model
from src.data_processing.dataloader import get_tabular_dl_dict
from src.training.train import train_model_w_best_param_tabular
from src.models.rue.predicting import get_all_predictions
from src.evaluation.evaluate import get_model_performance_tabular
from seed_file import seed
# seed = 2024

batch_size = 32
eval_batch_size = 128


tuning_seed = 2024

fp = FilePath(data_name=data_name, seed=seed)
fp_data_file = join(fp.get_preprocessed_folder(), "raw_data", "lungcancerdataset.csv")
fp_scaler_file = join(fp.get_preprocessed_folder(), f"minmax_scaler.pickle")
fp_encoder_file = join(fp.get_preprocessed_folder(), f"encoder.pickle")
fp_split_dict_file = join(fp.get_preprocessed_folder(), f"split_dict.joblib")
fp_split_dict_oversampled_file = join(fp.get_preprocessed_folder(), f"split_dict_oversampled.joblib")

# Load Data

In [ ]:
split_dict_scaled = joblib.load(fp_split_dict_file)
feat_cols_w_pc = ['pc1', 'pc2', 'pc3', 'age_interview', 'BMI', 'telomere length', 'Leisure screen time', 'ahei2010score', 'amed', 'dash', 'SBP', 'DBP', 'pgs000070', 'pgs000721', 'sex (0=Male, 1=Female)', 'alcohol_0_12', 'smoke_ex(1)', 'smoke_current(2)', 'prevalent diabetes']
target_col = "lung cancer"

# Classifier

## Tuning

In [ ]:
if seed == tuning_seed:
    pos_weight = get_pos_weight(split_dict_scaled, target_col)
    fp_model = join(fp.get_parent_folder(fn_model), "classifier_tuning.pt")
    fp_history = join(fp.get_parent_folder(fn_model), "classifier_tuning_history.jpg")
    train_param_dict = dict(max_epochs=500, lr=0.001, weight_decay=0.005, patience=5)
    classifier_tuning_df, classifier_best_param = tune_model(
        ModelClass=AE_Predictor, param_grid=dict(
            num_encoder_layers=[2, 3, 4], encoder_width=[128, 256, 512], 
            num_decoder_layers=[3], decoder_width=[128]
        ), 
        seed=seed,
        feature_cols=feat_cols_w_pc, target_col=target_col, 
        train_param_dict=train_param_dict, train_model_func=train_classifier, 
        split_dict=split_dict_scaled, pytorch_split_dict_func=get_tabular_dl_dict, class_weight=pos_weight, # Split dict is a dictionary of dataframes, pytorch_split_dict_func converts these dfs to dl
        fp_model=fp_model, fp_history=fp_history, 
        batch_size=batch_size, eval_batch_size=eval_batch_size,
        metric_to_monitor = "ce loss", maximise=False, # Metric used for Tuning
    )
    fp_tuning_classifier_file = join(fp.get_parent_folder("tuning"), "tuning_classifier.csv")
    classifier_tuning_df.to_csv(fp_tuning_classifier_file)
    display(classifier_tuning_df)
    print(classifier_best_param)

## Training

In [ ]:
fp_model = join(fp.get_parent_folder(fn_model), "classifier_tuned.pt")
fp_history = join(fp.get_parent_folder(fn_model), "classifier_tuned_history.jpg")
train_param_dict = dict(max_epochs=500, lr=0.001, weight_decay=0.005, patience=5)
classifier_best_param={'decoder_width': 128, 'encoder_width': 256, 'num_decoder_layers': 3, 'num_encoder_layers': 2}
classifier = train_model_w_best_param_tabular(
    ModelClass=AE_Predictor, best_param=classifier_best_param, 
    feature_cols=feat_cols_w_pc, target_col=target_col, 
    train_param_dict=train_param_dict, train_model_func=train_classifier, 
    split_dict=split_dict_scaled, pytorch_split_dict_func=get_tabular_dl_dict, # Split dict is a dictionary of dataframes, pytorch_split_dict_func converts these dfs to dl
    fp_model=fp_model, fp_history=fp_history, 
    batch_size=batch_size, eval_batch_size=eval_batch_size,
    metric_to_monitor="ce loss", maximise=False, class_weight=pos_weight, seed=seed
)

# Prediction + Performance Evaluation (MC)

In [ ]:
split_dict_pytorch = get_tabular_dl_dict(
    **split_dict_scaled, feat_cols=feat_cols_w_pc, target_col=target_col, shuffle_train=False,
    batch_size=batch_size, eval_batch_size=eval_batch_size
)
pred_df_classifier = get_all_predictions(
    model=classifier, **split_dict_pytorch, 
    feature_cols=feat_cols_w_pc, target_col=target_col, seed=seed
)
perf_df = get_model_performance_tabular(all_pred_df=pred_df_classifier, target_col=target_col)
perf_df

In [ ]:
fp_classifier_perf_file = join(fp.get_parent_folder(fn_pred_perf), "egRUE.csv")
perf_df.to_csv(fp_classifier_perf_file)
fp_classifier_predictions_file = join(fp.get_parent_folder(fn_pred), "tuning_classifier.csv")
pred_df_classifier.to_csv(fp_classifier_predictions_file)

# Decoder

## Tuning

In [ ]:
classifier_best_param={'decoder_width': 128, 'encoder_width': 256, 'num_decoder_layers': 3, 'num_encoder_layers': 2}
fp_model = join(fp.get_parent_folder(fn_model),"classifier_tuned.pt")
classifier = torch.load(fp_model, mmap=device)

In [ ]:
if seed==tuning_seed:
    # Interesting finding: Using L1Loss + Greater Capacity + Weight decay** + Non-Oversampled Improves Performance of RUE 
    fp_model = join(fp.get_parent_folder(fn_model), "decoder_tuning.pt")
    fp_history = join(fp.get_parent_folder(fn_model), "decoder_tuning_history.jpg")
    train_param_dict = dict(max_epochs=500, lr=0.00005, weight_decay=0.01, patience=5)
    decoder_tuning_df, decoder_best_param = tune_model(
        ModelClass=AE_Predictor, param_grid=dict(
            num_encoder_layers=[classifier_best_param["num_encoder_layers"]], 
            encoder_width=[classifier_best_param["encoder_width"]], 
            num_decoder_layers=[4, 5, 6], decoder_width=[256, 512, 1024]
        ), 
        seed=seed, batch_size=batch_size, eval_batch_size=eval_batch_size,
        feature_cols=feat_cols_w_pc, target_col=target_col, 
        train_param_dict=train_param_dict, train_model_func=train_decoder, 
        split_dict=split_dict_scaled, pytorch_split_dict_func=get_tabular_dl_dict, # Split dict is a dictionary of dataframes, pytorch_split_dict_func converts these dfs to dl
        fp_model=fp_model, fp_history=fp_history, 
        metric_to_monitor = "rue correlation", maximise=True, # Metric used for Tuning
        prev_model=classifier
    )
    fp_tuning_decoder_file = join(fp.get_parent_folder("tuning"), "tuning_decoder.csv")
    decoder_tuning_df.to_csv(fp_tuning_decoder_file)
    display(decoder_tuning_df)
    print(decoder_best_param)

## Training

In [ ]:
fp_model = join(fp.get_parent_folder(fn_model),"decoder_tuned.pt")
fp_history = join(fp.get_parent_folder(fn_model), "decoder_tuned_history.jpg")
decoder_best_param = {'decoder_width': 1024, 'encoder_width': 256, 'num_decoder_layers': 5, 'num_encoder_layers': 2}
train_param_dict = dict(max_epochs=500, lr=0.00005, weight_decay=0.01, patience=5)
classifier_decoder = train_model_w_best_param_tabular(
    ModelClass=AE_Predictor, best_param=decoder_best_param, 
    feature_cols=feat_cols_w_pc, target_col=target_col, 
    train_param_dict=train_param_dict, train_model_func=train_decoder, 
    split_dict=split_dict_scaled, pytorch_split_dict_func=get_tabular_dl_dict, # Split dict is a dictionary of dataframes, pytorch_split_dict_func converts these dfs to dl
    seed=seed, batch_size=batch_size, eval_batch_size=eval_batch_size,
    fp_model=fp_model, fp_history=fp_history, 
    metric_to_monitor="rue correlation", maximise=True,
    prev_model=classifier
)

# Prediction + Performance Evaluation (MC)

In [ ]:
split_dict_pytorch = get_tabular_dl_dict(
    **split_dict_scaled, feat_cols=feat_cols_w_pc, target_col=target_col, shuffle_train=False,
     batch_size=batch_size, eval_batch_size=eval_batch_size
)
pred_df_classifier_decoder = get_all_predictions(
    model=classifier_decoder, **split_dict_pytorch, 
    feature_cols=feat_cols_w_pc, target_col=target_col, T=10, seed=seed
)
get_model_performance_tabular(all_pred_df=pred_df_classifier_decoder, target_col=target_col)

In [ ]:
fp_decoder_predictions_file = join(fp.get_parent_folder(fn_pred), "tuning_decoder.csv")
pred_df_classifier_decoder.to_csv(fp_decoder_predictions_file)